In [1]:
import numpy as np
import numpy as np
from scipy.sparse.linalg import eigsh
from scipy.sparse import lil_matrix

In [2]:
class spin_state():
    """
    This is the spin state class
    """
    def __init__(self, L,i):
        """

        Parameters
        ----------
        L : int 
            Length of chain.
        i : int
            The number that shall be encoded into binary representation 
            as a spin state

        Returns
        -------
        None.

        """
        self.state=np.zeros(L, dtype=int)
        self.maxval=0
        self.id=i
        self.basis_pos=-1

        for j in np.arange(0,L,1):
            self.maxval+=2**(j)
        self.nr=i
        if(i>self.maxval):
            print("i to large")
            return
        i_cp=i
        index=0
        while i_cp>0:
            if(i_cp%2!=0):
                self.state[L-1-index]=1
            i_cp//=2
            index+=1
        self.mag=np.sum(self.state)
        return

    def set_basis_pos(self, x):
        """
        

        Parameters
        ----------
        x : int
            The position of the state in a basis.

        Returns
        -------
        None.

        """
        self.basis_pos = x 
        return
    
    
    


In [3]:
def make_spinbasis(L, conserve_M=False, M=0):
    """
    Function to generate the spin basis

    Parameters
    ----------
    L : int
        Length of chain.
    conserve_M : (bool), optional
        DESCRIPTION. The default is False. Set True if magnetization 
        is conserved.
    M : int, optional
        DESCRIPTION. The default is 0. Magnetization. Only matters if 
        conserve_M=True
    Returns
    -------
    states : (dict)
        Dictionary containing the spin basis states as values and the decimal 
        representation.

    """
    
    
    states={}
    # the position of the state in the basis
    state_nr=0
    for i in np.arange(0, 2**L,1):
        state=spin_state(L, i)
        # print(state.state, state.mag)
        if(conserve_M):
            if(state.mag==M):
                
                state.set_basis_pos(state_nr)
                states[state.id]=state
                state_nr+=1
        else:
            state.set_basis_pos(state_nr)
            states[state.id]=state
            state_nr+=1
    return states

In [4]:
def convert_array_to_number(arr):
    """
    This function converts an array of spins e.g. [0,1,1,0]
    into their corresponding integer representation (each array is encoded as the binary representation
                                                     of an integer)
    Parameters
    ----------
    arr : numpy array
    Takes a numpy array with the elements 0,1 

    Returns
    -------
    nr : int
        The integer from which the arr is the binary representation

    """
    nr=0
    j=0
    for i in np.flip(arr):
        nr+=i*2**(j)
        j+=1
    return nr

In [11]:
def generate_fermi_ekin(states,L, PB=True, use_sparse=False):
    """
    This function generates the XXZ-Hamiltonian with periodic boundary condictions.
    
    H=\sum\limits_{i=0}^{L-1}( J (S_i^{x}S^{x}_{i+1} +S_i^{y}S^{y}_{i+1})+
                              \Delta (S_i^{z}\S^{z}_{i+1})+hS^{z}_i)

    Parameters
    ----------
    states : TYPE
        DESCRIPTION.
    J : float
        The spin-spin interaction.
    h : float
        Magnetic field.
    L : int
        Length of chain.
    PB : bool
        Indicating whether to use periodic boundary conditions

    Returns
    -------
    H : numpy array
        Hamiltonian of the system.

    """
    end_val=L
    if PB==False:
        end_val=L-1
    if use_sparse:
        H=lil_matrix((len(states),len(states)))
    else:
        H=np.zeros((len(states),len(states)))        
    for state in states.values():
        for i in np.arange(0,end_val,1):
            j=(i+1)%L
            if state.state[i]==state.state[j]:
                H[state.basis_pos,state.basis_pos]+=1/4
            else:
                H[state.basis_pos,state.basis_pos]-=1/4
                arr=np.copy(state.state)
                arr_cp=np.copy(state.state)
                arr[i]=arr_cp[j]
                arr[j]=arr_cp[i]
                H[states[convert_array_to_number(arr)].basis_pos,state.basis_pos]+=1./2

                
    return H



In [20]:
L=7
basis=make_spinbasis(L, conserve_M=False, M=int(L/2))

In [21]:
H=generate_fermi_ekin(basis,L, PB=True)

In [22]:
e,v=np.linalg.eigh(H)

In [23]:
e[0]/L

np.float64(-0.4078827509780992)